In [ ]:
# Mount Google Drive
from google.colab import drive  # access drive
drive.mount('/content/drive')

# pandas, numpy for data handling
import pandas as pd
import numpy as np

# path
path = "/content/drive/MyDrive/BigData/assignment3/ml-latest-small/"

# load data
ratings = pd.read_csv(path + "ratings.csv")
movies = pd.read_csv(path + "movies.csv")

ratings.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
# create user-item matrix
user_item = ratings.pivot(index="userId", columns="movieId", values="rating")

# fill missing values with user mean
user_item_filled = user_item.apply(lambda row: row.fillna(row.mean()), axis=1)

# convert to numpy
R = user_item_filled.values

In [ ]:
user_item.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print("R (sample):")
print(R[:5, :5])

R (sample):
[[4.         4.36637931 4.         4.36637931 4.36637931]
 [3.94827586 3.94827586 3.94827586 3.94827586 3.94827586]
 [2.43589744 2.43589744 2.43589744 2.43589744 2.43589744]
 [3.55555556 3.55555556 3.55555556 3.55555556 3.55555556]
 [4.         3.63636364 3.63636364 3.63636364 3.63636364]]


In [ ]:
# subtract user mean
R_mean = np.mean(R, axis=1).reshape(-1, 1)
R_norm = R - R_mean

In [ ]:
R_mean.shape

(610, 1)

In [ ]:
R_mean[0:5]

array([[4.36637931],
       [3.94827586],
       [2.43589744],
       [3.55555556],
       [3.63636364]])

In [ ]:
R_norm.shape

(610, 9724)

In [ ]:
R_norm[0:5, 0:5]

array([[-3.66379310e-01,  8.88178420e-16, -3.66379310e-01,
         8.88178420e-16,  8.88178420e-16],
       [-1.33226763e-15, -1.33226763e-15, -1.33226763e-15,
        -1.33226763e-15, -1.33226763e-15],
       [-4.44089210e-16, -4.44089210e-16, -4.44089210e-16,
        -4.44089210e-16, -4.44089210e-16],
       [-1.33226763e-15, -1.33226763e-15, -1.33226763e-15,
        -1.33226763e-15, -1.33226763e-15],
       [ 3.63636364e-01,  1.33226763e-15,  1.33226763e-15,
         1.33226763e-15,  1.33226763e-15]])

In [ ]:
from scipy.sparse.linalg import svds  # SVD for large matrix

# choose number of latent factors
k = 50

# perform SVD
U, sigma, Vt = svds(R_norm, k=k)

# convert sigma to diagonal matrix
sigma = np.diag(sigma)

In [ ]:
U.shape

(610, 50)

In [ ]:
sigma.shape,  sigma[:5,:5]

((50, 50),
 array([[20.52281041,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        , 20.66212229,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        , 20.91516737,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , 20.99892796,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        , 21.20891333]]))

In [ ]:
U.shape, U[:5,:5]

((610, 50),
 array([[ 2.97402022e-02,  5.88769126e-02, -2.15642139e-02,
          1.97126858e-02, -1.16541849e-04],
        [-2.86112252e-03,  6.64369792e-05, -9.38051346e-04,
          2.53887681e-03, -3.27048527e-03],
        [-1.02969292e-02,  1.74163031e-02, -9.68613144e-04,
         -7.96450087e-03,  6.15724265e-03],
        [-2.22600260e-01,  1.51439282e-01,  9.99614702e-02,
         -5.45468670e-02,  8.50223400e-02],
        [ 5.23072517e-03, -1.29519677e-02, -2.92353605e-03,
         -9.92165189e-03,  8.47883109e-04]]))

In [ ]:
Vt.shape, Vt[:5,:5]

((50, 9724),
 array([[-0.03090522,  0.00349103,  0.01632729,  0.00542772, -0.00661819],
        [-0.03298506,  0.00216567,  0.04372385,  0.00162871, -0.00709538],
        [-0.05095091,  0.0211382 ,  0.01175884, -0.00784575, -0.00080405],
        [-0.09578479, -0.007686  ,  0.00871357,  0.00435521, -0.0176919 ],
        [-0.01571867,  0.00686237, -0.00273076, -0.00165812, -0.00672678]]))

In [ ]:
# reconstruct matrix
R_pred = np.dot(np.dot(U, sigma), Vt) + R_mean

# clip ratings to [1, 5]
R_pred = np.clip(R_pred, 1, 5)

In [ ]:
# first 5 users, first 5 movies

print("R (sample):")
print(R[:5, :5])

print("\nR_pred (sample):")
print(R_pred[:5, :5])

R (sample):
[[4.         4.36637931 4.         4.36637931 4.36637931]
 [3.94827586 3.94827586 3.94827586 3.94827586 3.94827586]
 [2.43589744 2.43589744 2.43589744 2.43589744 2.43589744]
 [3.55555556 3.55555556 3.55555556 3.55555556 3.55555556]
 [4.         3.63636364 3.63636364 3.63636364 3.63636364]]

R_pred (sample):
[[4.30616055 4.29639677 4.40595086 4.370563   4.3429083 ]
 [3.94598477 3.93225963 3.9367516  3.95325602 3.95645634]
 [2.42520627 2.4731044  2.40444357 2.44225211 2.46269148]
 [3.5913914  3.54279123 3.56774337 3.50129649 3.61802831]
 [3.82692331 3.62317845 3.61951202 3.61841249 3.60747452]]


In [ ]:
from sklearn.metrics import mean_squared_error

def reconstruction_rmse():

    preds = []
    actuals = []

    for row in ratings.itertuples():
        user_id = row.userId
        movie_id = row.movieId
        rating = row.rating

        user_idx = user_item.index.get_loc(user_id)
        movie_idx = user_item.columns.get_loc(movie_id)

        preds.append(R_pred[user_idx, movie_idx])
        actuals.append(rating)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    return rmse

print("Reconstruction RMSE:", reconstruction_rmse())

Reconstruction RMSE: 0.6198874462513957


In [ ]:
recommend_svd(1, top_k=10)

,movieId,title
302,344,Ace Ventura: Pet Detective (1994)
474,541,Blade Runner (1982)
659,858,"Godfather, The (1972)"
945,1246,Dead Poets Society (1989)
1711,2300,"Producers, The (1968)"
2110,2804,"Christmas Story, A (1983)"
2996,4011,Snatch (2000)
3638,4993,"Lord of the Rings: The Fellowship of the Ring,..."
4137,5952,"Lord of the Rings: The Two Towers, The (2002)"
4800,7153,"Lord of the Rings: The Return of the King, The..."


In [ ]:
from sklearn.metrics import mean_squared_error

def compute_rmse():

    preds = []
    actuals = []

    for row in ratings.itertuples():
        user_id = row.userId
        movie_id = row.movieId
        rating = row.rating

        user_idx = user_item.index.get_loc(user_id)
        movie_idx = user_item.columns.get_loc(movie_id)

        preds.append(R_pred[user_idx, movie_idx])
        actuals.append(rating)

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    return rmse

print("SVD RMSE:", compute_rmse())

SVD RMSE: 0.6198874462513957


In [ ]:
def evaluate_svd(user_id, k=10):

    recs = recommend_svd(user_id, k)["movieId"].tolist()

    relevant = set(
        ratings[
            (ratings["userId"] == user_id) & (ratings["rating"] >= 4)
        ]["movieId"]
    )

    intersection = set(recs).intersection(relevant)

    precision = len(intersection) / k
    recall = len(intersection) / len(relevant) if len(relevant) > 0 else 0

    print(f"\nUser {user_id}")
    print("Intersection:", list(intersection))
    print(f"Precision@{k}: {precision:.3f}")
    print(f"Recall@{k}: {recall:.3f}")

In [ ]:
evaluate_svd(1, k=50)


User 1
Intersection: []
Precision@50: 0.000
Recall@50: 0.000


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# user similarity
user_sim = cosine_similarity(user_item_filled)

def recommend_user_cf(user_id, top_k=10):

    user_idx = user_item.index.get_loc(user_id)

    sim_scores = user_sim[user_idx]

    weighted_ratings = np.dot(sim_scores, user_item_filled.values)
    norm = np.sum(np.abs(sim_scores))

    preds = weighted_ratings / norm

    rated = set(ratings[ratings["userId"] == user_id]["movieId"])

    movie_scores = list(zip(user_item.columns, preds))
    movie_scores = [m for m in movie_scores if m[0] not in rated]

    movie_scores.sort(key=lambda x: x[1], reverse=True)

    top_movies = [m[0] for m in movie_scores[:top_k]]

    return top_movies

In [ ]:
# item similarity
item_sim = cosine_similarity(user_item_filled.T)

def recommend_item_cf(user_id, top_k=10):

    user_idx = user_item.index.get_loc(user_id)

    user_ratings = user_item_filled.values[user_idx]

    preds = np.dot(item_sim, user_ratings)

    rated = set(ratings[ratings["userId"] == user_id]["movieId"])

    movie_scores = list(zip(user_item.columns, preds))
    movie_scores = [m for m in movie_scores if m[0] not in rated]

    movie_scores.sort(key=lambda x: x[1], reverse=True)

    top_movies = [m[0] for m in movie_scores[:top_k]]

    return top_movies

In [ ]:
print("SVD RMSE:", compute_rmse())

print("\nSVD:")
evaluate_svd(1, 10)

print("\nUser CF:", recommend_user_cf(1, 10))
print("\nItem CF:", recommend_item_cf(1, 10))

SVD RMSE: 0.6198874462513957

SVD:

User 1
Intersection: []
Precision@10: 0.000
Recall@10: 0.000

User CF: [318, 858, 4993, 7153, 589, 4226, 58559, 1221, 5952, 32]

Item CF: [2820, 3568, 7260, 32799, 43744, 55844, 68835, 69495, 92954, 132454]
